In [ ]:
#@title Install Dependencies
!sudo apt update
!sudo apt-get install zstd
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
64 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [ ]:
#@title Configure Ollama subprocess
import threading
import subprocess
import time
import os

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

The model selected by default gemma4:eb4 is very lightweight and can run on mobile slowly or near instantly on a powerful machine. Feel free to browse https://ollama.com/library and replace model_name to test models but for the purpose of this project we will likely need a vision model (if at all).

In [ ]:
#@title Download Model
model_name = "gemma4:e4b" #@param {type:"string"}
os.system(f"ollama pull {model_name}")

0

Upload any image to test

In [ ]:
#@title Upload sample image (This expects you press the upload button when it gets to this point in execution)
from google.colab import files
from pathlib import Path
if Path("image.jpg").exists():
  os.remove("image.jpg")
uploaded = files.upload()
os.rename((list(uploaded.keys())[0]),'image.jpg')

Saving OIP-3572450166.jpg to OIP-3572450166.jpg


This is an example of a crude implementation. It has forced json output via FSM (this part isnt vibes) but the LLM's usage of the json can vary somewhat. This can be mitagated by prompting. Production would likely use a quantized version which would reduce resource usage.

The first run will take a while due to having to allocate large amounts of memory. After the time for execution is near instant. This will also be reduced as a problem with quantization

In [ ]:
#@title API request (production would use an internal library)
import requests
import base64

image_path = "/content/image.jpg"
prompt = "Describe the object in a single word with the following categories, \"Dog, Cat or Other\" labed as 'Category'. Do not respond further." #@param {type:"string"}

with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": model_name,
        "prompt": prompt,
        "images": [image_b64],
        "format": "json",
        "stream": False,
    }
)

print(response.json())

{'model': 'gemma4:e4b', 'created_at': '2026-06-03T16:25:50.212920244Z', 'response': '{"Category": "Other"}', 'done': True, 'done_reason': 'stop', 'context': [2, 105, 9731, 107, 98, 107, 106, 107, 105, 2364, 107, 236840, 3024, 236772, 236771, 236842, 54234, 506, 2495, 528, 496, 3161, 3658, 607, 506, 2269, 505, 805, 679, 695, 21871, 236764, 13953, 653, 7067, 4796, 524, 618, 756, 234009, 6748, 3574, 711, 8932, 3342, 236761, 106, 107, 105, 4368, 107, 14937, 12962, 1083, 623, 13517, 25938], 'total_duration': 1612497397, 'load_duration': 1259741750, 'prompt_eval_count': 308, 'prompt_eval_duration': 155343000, 'eval_count': 7, 'eval_duration': 168454000}
